In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import IncrementalPCA


# ============================================================
# CONFIGURAÇÃO
# ============================================================

BASE_DIR = Path.home() / "Documentos" / "PUC" / "Projeto_Deep_Learning"

DATA_DIR = BASE_DIR / "data"

OUTPUT_DIR = (
    BASE_DIR
    / "Embeddings"
    / "DL"
    / "Texto"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLITS = [
    "train",
    "validation",
    "test"
]

TEXT_COLUMN = "input"

N_COMPONENTS = 128

MODEL_DIM = 1024

BATCH_SIZE = 32

DEVICE = (
    "cuda"
    if __import__("torch").cuda.is_available()
    else "cpu"
)


# ============================================================
# MODELO
# ============================================================

MODEL_NAME = "Shitao/bge-m3"

print("=" * 60)
print("CARREGANDO BGE-M3")
print("=" * 60)

print(f"Device: {DEVICE}")

model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE
)

print("Modelo carregado.")


# ============================================================
# LOCALIZAR CSV
# ============================================================

def encontrar_csv(split):

    caminho = DATA_DIR / f"{split}.csv"

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: "
            f"{caminho}"
        )

    return caminho


# ============================================================
# CARREGAR DATASETS
# ============================================================

dfs = {}

for split in SPLITS:

    csv_path = encontrar_csv(split)

    print(
        f"\nCarregando {split}: "
        f"{csv_path}"
    )

    df = pd.read_csv(csv_path)

    colunas_obrigatorias = [
        "patch_id",
        TEXT_COLUMN
    ]

    for coluna in colunas_obrigatorias:

        if coluna not in df.columns:

            raise ValueError(
                f"{csv_path} não possui "
                f"a coluna '{coluna}'."
            )

    df = df[
        [
            "patch_id",
            TEXT_COLUMN
        ]
    ].copy()

    df[TEXT_COLUMN] = (
        df[TEXT_COLUMN]
        .fillna("")
        .astype(str)
    )

    dfs[split] = df

    print(
        f"Patches: "
        f"{len(df):,}"
    )


# ============================================================
# GERAR EMBEDDINGS 1024D
# ============================================================

print("\n" + "=" * 60)
print("GERANDO EMBEDDINGS BGE-M3 — 1024D")
print("=" * 60)

embeddings_1024 = {}

for split in SPLITS:

    print(
        f"\nProcessando: "
        f"{split.upper()}"
    )

    textos = (
        dfs[split][TEXT_COLUMN]
        .tolist()
    )

    X = model.encode(
        textos,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    if X.shape[1] != MODEL_DIM:

        raise ValueError(
            f"Embedding inesperado: "
            f"{X.shape}"
        )

    embeddings_1024[split] = (
        X.astype(np.float32)
    )

    print(
        f"Shape: "
        f"{X.shape}"
    )


# ============================================================
# PCA
# ============================================================

print("\n" + "=" * 60)
print("PCA BGE-M3: 1024D → 128D")
print("=" * 60)

pca = IncrementalPCA(
    n_components=N_COMPONENTS,
    batch_size=256
)

X_train = embeddings_1024["train"]

for inicio in range(
    0,
    len(X_train),
    256
):

    fim = min(
        inicio + 256,
        len(X_train)
    )

    if fim - inicio < N_COMPONENTS:
        break

    pca.partial_fit(
        X_train[inicio:fim]
    )

print(
    "PCA ajustado somente no train."
)


# ============================================================
# TRANSFORMAR E SALVAR
# ============================================================

for split in SPLITS:

    X = pca.transform(
        embeddings_1024[split]
    )

    result = pd.DataFrame(
        X.astype(np.float32),
        columns=[
            f"dim_{i:03d}"
            for i in range(N_COMPONENTS)
        ]
    )

    result.insert(
        0,
        "patch_id",
        dfs[split]["patch_id"].values
    )

    output_path = (
        OUTPUT_DIR
        / f"{split}.parquet"
    )

    result.to_parquet(
        output_path,
        index=False
    )

    print(
        f"{split}: "
        f"{result.shape}"
    )

    print(
        f"Salvo em: "
        f"{output_path}"
    )


print("\n" + "=" * 60)
print("PIPELINE DE TEXTO CONCLUÍDO")
print("=" * 60)

/home/ettore/miniconda3/envs/datascience/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CARREGANDO BGE-M3
Device: cuda
Modelo carregado.

Carregando train: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/train.csv
Patches: 20,996

Carregando validation: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/validation.csv
Patches: 4,500

Carregando test: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/data/test.csv
Patches: 4,504

GERANDO EMBEDDINGS BGE-M3 — 1024D

Processando: TRAIN


Batches: 100%|██████████| 657/657 [01:17<00:00,  8.46it/s]


Shape: (20996, 1024)

Processando: VALIDATION


Batches: 100%|██████████| 141/141 [00:15<00:00,  9.12it/s]


Shape: (4500, 1024)

Processando: TEST


Batches: 100%|██████████| 141/141 [00:15<00:00,  9.13it/s]


Shape: (4504, 1024)

PCA BGE-M3: 1024D → 128D
PCA ajustado somente no train.
train: (20996, 129)
Salvo em: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/Embeddings/DL/Texto/train.parquet
validation: (4500, 129)
Salvo em: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/Embeddings/DL/Texto/validation.parquet
test: (4504, 129)
Salvo em: /home/ettore/Documentos/PUC/Projeto_Deep_Learning/Embeddings/DL/Texto/test.parquet

PIPELINE DE TEXTO CONCLUÍDO


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
import torch

from transformers import AutoModel, AutoImageProcessor
from sklearn.decomposition import IncrementalPCA


# ============================================================
# CONFIGURAÇÃO
# ============================================================

BASE_DIR = Path.home() / "Documentos" / "PUC" / "Projeto_Deep_Learning"

DATA_DIR = BASE_DIR / "data"

IMAGE_DIR = DATA_DIR / "BigEarthNet-S2-10bands-96"

OUTPUT_DIR = (
    BASE_DIR
    / "Embeddings"
    / "DL"
    / "Imagem"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SPLITS = [
    "train",
    "validation",
    "test"
]

BANDAS = [
    "B02",
    "B03",
    "B04",
    "B05",
    "B06",
    "B07",
    "B08",
    "B8A",
    "B11",
    "B12"
]

N_COMPONENTS = 128

# Embedding original do SatMAE-PP
MODEL_DIM = 1024

# Quantidade de imagens por batch
BATCH_SIZE = 8

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================================================
# MODELO
# ============================================================

MODEL_NAME = "BiliSakura/SATMAE-PP-transformers"

SUBFOLDER = (
    "satmae-pp-vit-large-patch8-fmow-sentinel-pretrain"
)

print("=" * 60)
print("CARREGANDO SATMAE-PP")
print("=" * 60)

print(f"Device: {DEVICE}")

processor = AutoImageProcessor.from_pretrained(
    MODEL_NAME,
    subfolder=SUBFOLDER,
    trust_remote_code=True
)

model = AutoModel.from_pretrained(
    MODEL_NAME,
    subfolder=SUBFOLDER,
    trust_remote_code=True
)

model = model.to(DEVICE)
model.eval()

print("Modelo carregado.")


# ============================================================
# LOCALIZAR CSV
# ============================================================

def encontrar_csv(split):

    caminho = DATA_DIR / f"{split}.csv"

    if not caminho.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {caminho}"
        )

    return caminho


# ============================================================
# CARREGAR IDS
# ============================================================

dfs = {}

for split in SPLITS:

    csv_path = encontrar_csv(split)

    print(
        f"\nCarregando {split}: "
        f"{csv_path}"
    )

    df = pd.read_csv(csv_path)

    if "patch_id" not in df.columns:
        raise ValueError(
            f"{csv_path} não possui "
            f"'patch_id'."
        )

    dfs[split] = df[
        ["patch_id"]
    ].copy()

    print(
        f"Patches: "
        f"{len(df):,}"
    )


# ============================================================
# CARREGAR UM PATCH
# ============================================================

def carregar_patch(
    split,
    patch_id
):

    patch_dir = (
        IMAGE_DIR
        / split
        / patch_id
    )

    if not patch_dir.exists():
        raise FileNotFoundError(
            f"Patch não encontrado:\n"
            f"{patch_dir}"
        )

    bandas = []

    for banda in BANDAS:

        arquivo = (
            patch_dir
            / f"{patch_id}_{banda}.tif"
        )

        if not arquivo.exists():
            raise FileNotFoundError(
                f"Banda não encontrada:\n"
                f"{arquivo}"
            )

        with rasterio.open(arquivo) as src:

            imagem = src.read(1)

        if imagem.shape != (96, 96):

            raise ValueError(
                f"{arquivo.name}: "
                f"{imagem.shape}; "
                f"esperado (96, 96)"
            )

        bandas.append(imagem)

    # (10, 96, 96)
    imagem = np.stack(
        bandas,
        axis=0
    )

    # SatMAE processor espera
    # (H, W, C)
    imagem = np.transpose(
        imagem,
        (1, 2, 0)
    )

    return imagem


# ============================================================
# GERAR EMBEDDING SATMAE
# ============================================================

def gerar_embeddings_imagem(
    split,
    patch_ids
):

    imagens = []

    for patch_id in patch_ids:

        imagem = carregar_patch(
            split,
            patch_id
        )

        imagens.append(imagem)

    # (B, 96, 96, 10)
    imagens = np.stack(
        imagens,
        axis=0
    )

    inputs = processor(
        images=list(imagens),
        return_tensors="pt",
        do_resize=False
    )

    inputs = {
        chave: valor.to(DEVICE)
        for chave, valor in inputs.items()
    }

    with torch.no_grad():

        outputs = model(
            **inputs
        )

    # Dependendo da implementação,
    # last_hidden_state terá:
    #
    # (B, tokens, 1024)
    #
    # Usamos média dos tokens caso
    # o modelo não disponibilize
    # diretamente o pooled output.

    if hasattr(
        outputs,
        "pooler_output"
    ) and outputs.pooler_output is not None:

        embeddings = (
            outputs.pooler_output
        )

    else:

        embeddings = (
            outputs.last_hidden_state
            .mean(dim=1)
        )

    return embeddings.cpu().numpy()


# ============================================================
# GERAR 1024D
# ============================================================

print("\n" + "=" * 60)
print("GERANDO EMBEDDINGS SATMAE-PP — 1024D")
print("=" * 60)

embeddings_1024 = {}

for split in SPLITS:

    print(
        f"\nProcessando: "
        f"{split.upper()}"
    )

    df = dfs[split]

    todos_embeddings = []

    for inicio in range(
        0,
        len(df),
        BATCH_SIZE
    ):

        fim = min(
            inicio + BATCH_SIZE,
            len(df)
        )

        patch_ids = (
            df["patch_id"]
            .iloc[inicio:fim]
            .tolist()
        )

        X = gerar_embeddings_imagem(
            split,
            patch_ids
        )

        if X.shape[1] != MODEL_DIM:

            raise ValueError(
                f"Embedding inesperado: "
                f"{X.shape}"
            )

        todos_embeddings.append(
            X.astype(np.float32)
        )

        print(
            f"\r{fim:,}/{len(df):,}",
            end=""
        )

    print()

    embeddings_1024[split] = np.vstack(
        todos_embeddings
    )

    print(
        f"Shape: "
        f"{embeddings_1024[split].shape}"
    )


# ============================================================
# PCA
# ============================================================

print("\n" + "=" * 60)
print("PCA SATMAE-PP: 1024D → 128D")
print("=" * 60)

pca = IncrementalPCA(
    n_components=N_COMPONENTS,
    batch_size=256
)

X_train = embeddings_1024["train"]

for inicio in range(
    0,
    len(X_train),
    256
):

    fim = min(
        inicio + 256,
        len(X_train)
    )

    # Proteção para último batch
    if fim - inicio < N_COMPONENTS:
        break

    pca.partial_fit(
        X_train[inicio:fim]
    )

print(
    "PCA ajustado."
)


# ============================================================
# TRANSFORMAR E SALVAR
# ============================================================

for split in SPLITS:

    X = pca.transform(
        embeddings_1024[split]
    )

    result = pd.DataFrame(
        X.astype(np.float32),
        columns=[
            f"dim_{i:03d}"
            for i in range(N_COMPONENTS)
        ]
    )

    result.insert(
        0,
        "patch_id",
        dfs[split]["patch_id"].values
    )

    output_path = (
        OUTPUT_DIR
        / f"{split}.parquet"
    )

    result.to_parquet(
        output_path,
        index=False
    )

    print(
        f"{split}: "
        f"{result.shape}"
    )

    print(
        f"Salvo em: "
        f"{output_path}"
    )


print("\n" + "=" * 60)
print("PIPELINE DE IMAGEM CONCLUÍDO")
print("=" * 60)

CARREGANDO SATMAE-PP
Device: cuda


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


OSError: BiliSakura/SATMAE-PP-transformers does not appear to have a file named image_processing_satmae_pp.py. Checkout 'https://huggingface.co/BiliSakura/SATMAE-PP-transformers/tree/main' for available files.

In [4]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_DIR = snapshot_download(
    repo_id="BiliSakura/SATMAE-PP-transformers",
    allow_patterns=[
        "satmae-pp-vit-large-patch8-fmow-sentinel-pretrain/*"
    ]
)

print(MODEL_DIR)

Fetching 6 files: 100%|██████████| 6/6 [00:35<00:00,  5.92s/it]

/home/ettore/.cache/huggingface/hub/models--BiliSakura--SATMAE-PP-transformers/snapshots/7d918c2ab86b92e72afd5992367d1801c6b886f4


In [5]:
from pathlib import Path

MODEL_PATH = (
    Path(MODEL_DIR)
    / "satmae-pp-vit-large-patch8-fmow-sentinel-pretrain"
)

print("Existe:", MODEL_PATH.exists())
print("Arquivos:")

for f in MODEL_PATH.iterdir():
    print(f.name)

Existe: True
Arquivos:
model.safetensors
config.json
image_processing_satmae_pp.py
pipeline_satmae_pp.py
preprocessor_config.json
modeling_satmae_pp.py


In [9]:
import torch

print("Torch:", torch.__version__)

# Não importe torchvision ainda
import importlib.metadata as metadata

print("Torchvision:", metadata.version("torchvision"))
print("Timm:", metadata.version("timm"))

Torch: 2.9.1+cu128
Torchvision: 0.29.0
Timm: 1.0.29


In [12]:
import torch
import torchvision
import timm

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Timm:", timm.__version__)

ImportError: cannot import name '_HAS_OPS' from 'torchvision.extension' (/home/ettore/miniconda3/envs/datascience/lib/python3.14/site-packages/torchvision/extension.py)